# Model preparation and train/test split

In [26]:
from Pose_Preprocessing_Pipeline_pasp import *

In [27]:
#example of coordintate output
window0 = X_clean[0]   # shape: (T, J, 3)
print(window0.shape)   # (60, 14, 3)


(60, 14, 3)


In [28]:
# get the patient_name_clean
# Extract video_id from window_ids_clean (assuming format: "video_id_winXXX_fYYY-ZZZ")
video_ids_clean = [w.split("_win")[0] for w in window_ids_clean]

# Build mapping from df_video: video_id -> patient_name
video_to_patient = dict(zip(df_video["video_id"], df_video["patient_name"]))

# Map window_ids to patient names
patient_names_clean = np.array([video_to_patient[vid] for vid in video_ids_clean])
print(f"Patient names for QC-clean windows: {patient_names_clean.shape}")
print(f"Unique patients: {len(np.unique(patient_names_clean))}")


Patient names for QC-clean windows: (15001,)
Unique patients: 409


In [62]:
# %%
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# -------------------------------
# Assume after QC you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# y_multilabel_clean: (N_windows, 5)
# window_ids_clean: list of window IDs
# patient_names_clean: list/array of patient_name per window
# -------------------------------

# Example:
# patient_names_clean = df_window['patient_name'].to_numpy()  

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

# Mask windows by patient
train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

# Apply masks
X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train  = y_multilabel_clean[train_mask]
y_ml_test   = y_multilabel_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. Separate abnormal windows for multi-label model
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

# -------------------------------
# 4. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)  # binary or multi-label

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]




Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
Multi-label train windows: 460, test windows: 355


In [ ]:
# Binary dataloaders
batch_size = 32
train_bin_loader = DataLoader(PoseDataset(X_train_stgcn, y_bin_train), batch_size=batch_size, shuffle=True)
test_bin_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=batch_size)

# Multi-label dataloaders
train_ml_loader = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_ml_loader  = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

### Define Model

In [33]:
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))

        # pool over time dimension
        self.pool = nn.AdaptiveAvgPool2d((1, num_joints))

        self.fc = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        # x is already (B, C, T, J)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        # Pool time dimension: (B, 128, 1, J)
        x = self.pool(x)

        # Flatten: (B, 128*J)
        x = x.flatten(1)

        return self.fc(x)


### Training Loop

In [34]:
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        if outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)  # binary
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    preds_list, targets_list = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            if outputs.shape[1] == 1:
                outputs = outputs.squeeze(1)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            preds_list.append(outputs.cpu())
            targets_list.append(y_batch.cpu())
    preds = torch.cat(preds_list)
    targets = torch.cat(targets_list)
    return total_loss / len(dataloader.dataset), preds, targets


In [35]:
#Train for binary model
device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_train_stgcn.shape[2]

binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_bin_loader, optimizer, criterion, device)
    val_loss, preds, targets = eval_model(binary_model, test_bin_loader, criterion, device)
    preds_label = (torch.sigmoid(preds) > 0.5).int()
    accuracy = (preds_label == targets.int()).float().mean()
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {accuracy:.4f}")


[Binary] Epoch 1/10 | Train Loss: 0.1255 | Test Acc: 0.9376
[Binary] Epoch 2/10 | Train Loss: 0.0476 | Test Acc: 0.9802
[Binary] Epoch 3/10 | Train Loss: 0.0367 | Test Acc: 0.9783
[Binary] Epoch 4/10 | Train Loss: 0.0225 | Test Acc: 0.9805
[Binary] Epoch 5/10 | Train Loss: 0.0205 | Test Acc: 0.9799
[Binary] Epoch 6/10 | Train Loss: 0.0181 | Test Acc: 0.9856
[Binary] Epoch 7/10 | Train Loss: 0.0165 | Test Acc: 0.9885
[Binary] Epoch 8/10 | Train Loss: 0.0162 | Test Acc: 0.9885
[Binary] Epoch 9/10 | Train Loss: 0.0148 | Test Acc: 0.9866
[Binary] Epoch 10/10 | Train Loss: 0.0135 | Test Acc: 0.9882


In [48]:
# Train for binary model with different weights
device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_train_stgcn.shape[2]

binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

# -------------------------------
# Class-weighted loss (IMPORTANT)
# -------------------------------
pos_weight = torch.tensor(
    [(len(y_bin_train) - y_bin_train.sum()) / y_bin_train.sum()],
    device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_bin_loader, optimizer, criterion, device)
    val_loss, preds, targets = eval_model(binary_model, test_bin_loader, criterion, device)
    preds_label = (torch.sigmoid(preds) > 0.5).int()
    accuracy = (preds_label == targets.int()).float().mean()
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {accuracy:.4f}")


[Binary] Epoch 1/10 | Train Loss: 0.9229 | Test Acc: 0.8897
[Binary] Epoch 2/10 | Train Loss: 0.4199 | Test Acc: 0.8420
[Binary] Epoch 3/10 | Train Loss: 0.2481 | Test Acc: 0.9687
[Binary] Epoch 4/10 | Train Loss: 0.1957 | Test Acc: 0.9683
[Binary] Epoch 5/10 | Train Loss: 0.1527 | Test Acc: 0.9856
[Binary] Epoch 6/10 | Train Loss: 0.1357 | Test Acc: 0.9885
[Binary] Epoch 7/10 | Train Loss: 0.1333 | Test Acc: 0.9853
[Binary] Epoch 8/10 | Train Loss: 0.0954 | Test Acc: 0.9783
[Binary] Epoch 9/10 | Train Loss: 0.0971 | Test Acc: 0.9469
[Binary] Epoch 10/10 | Train Loss: 0.0727 | Test Acc: 0.9859


In [36]:
multi_model = SimpleSTGCN(num_joints=num_joints, out_classes=5).to(device)
optimizer = torch.optim.Adam(multi_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(epochs):
    train_loss = train_model(multi_model, train_ml_loader, optimizer, criterion, device)
    val_loss, preds, targets = eval_model(multi_model, test_ml_loader, criterion, device)
    preds_bin = (torch.sigmoid(preds) > 0.5).int()
    
    # Compute per-label F1
    from sklearn.metrics import f1_score
    f1_per_label = f1_score(targets, preds_bin, average=None)
    f1_macro = f1_score(targets, preds_bin, average="macro")
    
    print(f"[Multi] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test F1 macro: {f1_macro:.4f} | per-label: {f1_per_label}")


/Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))


[Multi] Epoch 1/10 | Train Loss: 0.4915 | Test F1 macro: 0.0000 | per-label: [0. 0. 0. 0. 0.]
[Multi] Epoch 2/10 | Train Loss: 0.4459 | Test F1 macro: 0.0000 | per-label: [0. 0. 0. 0. 0.]
[Multi] Epoch 3/10 | Train Loss: 0.4205 | Test F1 macro: 0.0052 | per-label: [0.         0.         0.         0.02614379 0.        ]
[Multi] Epoch 4/10 | Train Loss: 0.3979 | Test F1 macro: 0.0000 | per-label: [0. 0. 0. 0. 0.]
[Multi] Epoch 5/10 | Train Loss: 0.3786 | Test F1 macro: 0.0322 | per-label: [0.         0.04379562 0.         0.11695906 0.        ]
[Multi] Epoch 6/10 | Train Loss: 0.3644 | Test F1 macro: 0.0441 | per-label: [0.         0.09395973 0.         0.12643678 0.        ]
[Multi] Epoch 7/10 | Train Loss: 0.3526 | Test F1 macro: 0.0438 | per-label: [0.         0.03007519 0.         0.18888889 0.        ]
[Multi] Epoch 8/10 | Train Loss: 0.3388 | Test F1 macro: 0.0732 | per-label: [0.05405405 0.02919708 0.         0.28272251 0.        ]
[Multi] Epoch 9/10 | Train Loss: 0.3356 | Test F

## Evaluate binary model

In [49]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

def evaluate_binary_model(model, dataloader, device):
    model.eval()

    all_logits = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch).squeeze(1)
            all_logits.append(logits.cpu())
            all_targets.append(y_batch.cpu())

    logits = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()

    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(targets, preds),
        "precision": precision_score(targets, preds, zero_division=0),
        "recall": recall_score(targets, preds, zero_division=0),
        "f1": f1_score(targets, preds, zero_division=0),
        "roc_auc": roc_auc_score(targets, probs),
        "confusion_matrix": confusion_matrix(targets, preds)
    }

    return metrics


In [50]:
# Evaluation after training
binary_metrics = evaluate_binary_model(binary_model, test_bin_loader, device)

for k, v in binary_metrics.items():
    print(f"{k}:\n{v}\n")


accuracy:
0.9859290054365206

precision:
0.9081364829396326

recall:
0.9746478873239437

f1:
0.9402173913043478

roc_auc:
0.9960835721399101

confusion_matrix:
[[2737   35]
 [   9  346]]



## Test data

In [51]:
def predict_binary(model, dataloader, device):
    model.eval()

    all_probs = []
    all_preds = []

    with torch.no_grad():
        for X_batch, _ in dataloader:   # labels not needed
            X_batch = X_batch.to(device)

            logits = model(X_batch).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs > 0.5).int()

            all_probs.append(probs.cpu())
            all_preds.append(preds.cpu())

    return torch.cat(all_probs), torch.cat(all_preds)


In [52]:
test_probs, test_preds = predict_binary(
    binary_model,
    test_bin_loader,
    device
)

print(test_probs.shape)  # (N_test_windows,)
print(test_preds.shape)  # (N_test_windows,)


torch.Size([3127])
torch.Size([3127])


In [53]:
patient_names_test = patient_names_clean[test_mask]
window_ids_test = window_ids_clean[test_mask]


In [54]:
import pandas as pd

pred_df = pd.DataFrame({
    "patient_name": patient_names_test,
    "window_id": window_ids_test,
    "prob_abnormal": test_probs.numpy(),
    "pred_abnormal": test_preds.numpy(),
    "true_label": y_bin_test
})

pred_df.head()


,patient_name,window_id,prob_abnormal,pred_abnormal,true_label
0,PA325,semantic_segmentation_PA325_UGS_WJ_2_DensePose...,0.001472,0,0
1,PA325,semantic_segmentation_PA325_UGS_WJ_2_DensePose...,0.004693,0,0
2,PA325,semantic_segmentation_PA325_UGS_WJ_2_DensePose...,0.002710,0,0
3,PA325,semantic_segmentation_PA325_UGS_WJ_2_DensePose...,0.001945,0,0
4,PA325,semantic_segmentation_PA325_UGS_WJ_2_DensePose...,0.000658,0,0


In [55]:
#Mean probability per patient
patient_pred = (
    pred_df
    .groupby("patient_name")["prob_abnormal"]
    .mean()
    .reset_index()
)

patient_pred["pred_abnormal"] = (patient_pred["prob_abnormal"] > 0.5).astype(int)
patient_pred


,patient_name,prob_abnormal,pred_abnormal
0,0,0.990536,1
1,17,0.998831,1
2,23,0.714320,1
3,40,0.219679,0
4,7,0.971731,1
...,...,...,...
77,PA368,0.000248,0
78,PA378,0.000685,0
79,PA380,0.002874,0
80,PA386,0.015425,0


In [56]:
#Majority vote
patient_pred = (
    pred_df
    .groupby("patient_name")["pred_abnormal"]
    .mean()
    .reset_index()
)

patient_pred["pred_abnormal"] = (patient_pred["pred_abnormal"] > 0.5).astype(int)


In [57]:
#Evalutate at patient level
true_patient_labels = (
    pred_df
    .groupby("patient_name")["true_label"]
    .max()  # if any window is abnormal → patient abnormal
)

from sklearn.metrics import accuracy_score, f1_score

print("Patient accuracy:", accuracy_score(true_patient_labels, patient_pred["pred_abnormal"]))
print("Patient F1:", f1_score(true_patient_labels, patient_pred["pred_abnormal"]))


Patient accuracy: 0.975609756097561
Patient F1: 0.8000000000000002


In [58]:
# Sanity check
print("Window positive rate:", test_preds.float().mean().item())
print("Avg abnormal prob:", test_probs.mean().item())
print("Patients predicted abnormal:", patient_pred["pred_abnormal"].mean())


Window positive rate: 0.12184201925992966
Avg abnormal prob: 0.12645922601222992
Patients predicted abnormal: 0.06097560975609756


In [59]:
# True patient abnormal rate
true_patient_rate = (
    pred_df
    .groupby("patient_name")["true_label"]
    .max()
    .mean()
)

print("True patient abnormal rate:", true_patient_rate)


True patient abnormal rate: 0.06097560975609756


In [60]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# ---------------------------
# 1. Put the model in eval mode
# ---------------------------
binary_model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_bin_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = binary_model(X_batch)
        # For BCEWithLogitsLoss, apply sigmoid
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs > 0.5).int()  # threshold at 0.5

        all_preds.append(preds.cpu())
        all_targets.append(y_batch.cpu().int())

# Concatenate all batches
y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_targets).numpy()

# ---------------------------
# 2. Compute metrics
# ---------------------------
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("Test Evaluation Metrics:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("Confusion Matrix:")
print(cm)


Test Evaluation Metrics:
Accuracy:  0.9859
Precision: 0.9081
Recall:    0.9746
F1 Score:  0.9402
ROC-AUC:   0.9810
Confusion Matrix:
[[2737   35]
 [   9  346]]


### Window-level prediction 
worse than before because of the downsampling of the normal data - don't use

In [61]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------------
# 1. Get window-level predictions on test set
# -----------------------------
binary_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_bin_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = binary_model(X_batch)
        if outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)
        preds = torch.sigmoid(outputs)  # convert logits to probabilities
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

# Flatten arrays
y_pred_windows = np.concatenate(all_preds)
y_true_windows = np.concatenate(all_targets)

# Convert probabilities to binary labels (threshold 0.5)
y_pred_windows_label = (y_pred_windows > 0.5).astype(int)

# -----------------------------
# 2. Aggregate to patient-level
# -----------------------------
# Assuming patient_names_test aligns with windows in test_bin_loader
# If not, collect patient names in the PoseDataset or DataLoader
patient_names_test = np.array(patient_names_clean)[test_mask]  # from your split earlier

unique_patients = np.unique(patient_names_test)
patient_true = []
patient_pred = []

for p in unique_patients:
    mask = patient_names_test == p
    # True: any abnormal window -> patient abnormal
    patient_true.append(int(y_true_windows[mask].any()))
    # Predicted: any predicted abnormal window -> patient abnormal
    patient_pred.append(int(y_pred_windows_label[mask].any()))

patient_true = np.array(patient_true)
patient_pred = np.array(patient_pred)

# -----------------------------
# 3. Confusion matrix & report
# -----------------------------
cm = confusion_matrix(patient_true, patient_pred)
print("Patient-level Confusion Matrix:")
print(cm)

print("\nPatient-level Classification Report:")
print(classification_report(patient_true, patient_pred, digits=4))

# -----------------------------
# 4. Optional: window-level metrics
# -----------------------------
cm_win = confusion_matrix(y_true_windows, y_pred_windows_label)
print("\nWindow-level Confusion Matrix:")
print(cm_win)


Patient-level Confusion Matrix:
[[69  8]
 [ 1  4]]

Patient-level Classification Report:
              precision    recall  f1-score   support

           0     0.9857    0.8961    0.9388        77
           1     0.3333    0.8000    0.4706         5

    accuracy                         0.8902        82
   macro avg     0.6595    0.8481    0.7047        82
weighted avg     0.9459    0.8902    0.9102        82


Window-level Confusion Matrix:
[[2737   35]
 [   9  346]]


In [63]:
# -------------------------------
# 2. Balance the training data (optional but recommended)
# -------------------------------
# Currently, abnormal windows are rare (~6%). We'll downsample normal windows.
abnormal_mask = y_bin_train == 1
normal_mask   = y_bin_train == 0

# Indices for abnormal and normal windows
abnormal_indices = np.where(abnormal_mask)[0]
normal_indices   = np.where(normal_mask)[0]

# Downsample normal windows to 2:1 ratio of normal:abnormal
np.random.seed(42)
normal_sample = np.random.choice(normal_indices, size=len(abnormal_indices)*2, replace=False)

# Combine indices
train_indices_balanced = np.concatenate([abnormal_indices, normal_sample])

# Apply balanced indices
X_train_stgcn_bal = X_train_stgcn[train_indices_balanced]
y_bin_train_bal   = y_bin_train[train_indices_balanced]

print(f"Balanced train windows: {X_train_stgcn_bal.shape[0]} (abnormal: {len(abnormal_indices)}, normal sampled: {len(normal_sample)})")

# -------------------------------
# 3. Define weighted loss to account for imbalance
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
pos_weight = torch.tensor(
    [(len(y_bin_train_bal) - y_bin_train_bal.sum()) / y_bin_train_bal.sum()],
    device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# 4. Create PyTorch DataLoaders using balanced training data
# -------------------------------
batch_size = 32

train_bin_loader = DataLoader(PoseDataset(X_train_stgcn_bal, y_bin_train_bal),
                              batch_size=batch_size, shuffle=True)
test_bin_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test),
                              batch_size=batch_size)

# Multi-label loaders remain the same
train_ml_loader = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_ml_loader  = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

# -------------------------------
# 5. Train the binary model
# -------------------------------
num_joints = X_train_stgcn.shape[2]
binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_bin_loader, optimizer, criterion, device)
    val_loss, preds, targets = eval_model(binary_model, test_bin_loader, criterion, device)
    preds_label = (torch.sigmoid(preds) > 0.5).int()
    accuracy = (preds_label == targets.int()).float().mean()
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {accuracy:.4f}")

# -------------------------------
# 6. Patient-level aggregation on test data
# -------------------------------
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, f1_score

binary_model.eval()
window_probs = []
with torch.no_grad():
    for X_batch, _ in test_bin_loader:
        X_batch = X_batch.to(device)
        probs = torch.sigmoid(binary_model(X_batch)).cpu().numpy()
        window_probs.extend(probs)

window_probs = np.array(window_probs).squeeze()
window_labels = y_bin_test
window_patients = patient_names_clean[test_mask]

# Patient-level prediction: abnormal if any window is predicted abnormal
threshold = 0.5
patient_preds = []
patient_true  = []

for patient in np.unique(window_patients):
    mask = window_patients == patient
    patient_preds.append((window_probs[mask] > threshold).any())
    patient_true.append((window_labels[mask] == 1).any())

patient_preds = np.array(patient_preds)
patient_true  = np.array(patient_true)

print("Patient-level Confusion Matrix:")
print(confusion_matrix(patient_true, patient_preds))
print("\nPatient-level Classification Report:")
print(classification_report(patient_true, patient_preds, digits=4))

# -------------------------------
# 7. Optional: Tune patient-level threshold for F1
# -------------------------------
best_thresh = 0.5
best_f1 = 0
for t in np.linspace(0.1, 0.9, 9):
    preds = []
    for patient in np.unique(window_patients):
        mask = window_patients == patient
        preds.append((window_probs[mask] > t).any())
    f1 = f1_score(patient_true, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
print(f"Best patient-level threshold: {best_thresh}, F1: {best_f1:.4f}")


Balanced train windows: 1380 (abnormal: 460, normal sampled: 920)
[Binary] Epoch 1/10 | Train Loss: 0.8605 | Test Acc: 0.9050
[Binary] Epoch 2/10 | Train Loss: 0.6695 | Test Acc: 0.9479
[Binary] Epoch 3/10 | Train Loss: 0.5394 | Test Acc: 0.9680
[Binary] Epoch 4/10 | Train Loss: 0.4039 | Test Acc: 0.9584
[Binary] Epoch 5/10 | Train Loss: 0.3107 | Test Acc: 0.9715
[Binary] Epoch 6/10 | Train Loss: 0.2434 | Test Acc: 0.8948
[Binary] Epoch 7/10 | Train Loss: 0.2130 | Test Acc: 0.9111
[Binary] Epoch 8/10 | Train Loss: 0.1898 | Test Acc: 0.9862
[Binary] Epoch 9/10 | Train Loss: 0.1526 | Test Acc: 0.9885
[Binary] Epoch 10/10 | Train Loss: 0.1422 | Test Acc: 0.9559
Patient-level Confusion Matrix:
[[49 28]
 [ 1  4]]

Patient-level Classification Report:
              precision    recall  f1-score   support

       False     0.9800    0.6364    0.7717        77
        True     0.1250    0.8000    0.2162         5

    accuracy                         0.6463        82
   macro avg     0.5525   

### Retrain removing downsampling

In [64]:
# -------------------------------
# 2a. Patient-level train/test split
# -------------------------------
import numpy as np
from sklearn.model_selection import train_test_split

# patient_names_clean is list/array of patient_name per window
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

# Mask windows by patient
train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

# Apply masks
X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]

# -------------------------------
# 2b. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))  # (N, 3, T, J)
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

print(f"Train shape: {X_train_stgcn.shape}, Test shape: {X_test_stgcn.shape}")


Train shape: (11874, 3, 60, 14), Test shape: (3127, 3, 60, 14)


In [ ]:
#Define Dataset and DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)  # BCEWithLogitsLoss expects float

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_loader = DataLoader(PoseDataset(X_train_stgcn, y_bin_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=batch_size)


In [75]:
# -------------------------------
# Re-define the model with global pooling
# -------------------------------
import torch.nn as nn
import torch.nn.functional as F

class SimpleSTGCN(nn.Module):
    def __init__(self, in_channels=3, out_classes=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.fc = nn.Linear(128, out_classes)  # after pooling

    def forward(self, x):
        # x: (batch, C, T, J)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # Global average pooling over time and joints
        x = x.mean(dim=[2,3])  # (batch, 128)
        x = self.fc(x)
        return x


In [77]:
#Setup training with pos_weight
device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_train_stgcn.shape[3]  # last dim is V (num joints)

binary_model = SimpleSTGCN(in_channels=3, out_classes=1).to(device)

optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

# Compute pos_weight for BCE to handle imbalance
pos_weight = torch.tensor(
    [(len(y_bin_train) - y_bin_train.sum()) / y_bin_train.sum()],
    device=device
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


In [78]:
#Training / evaluation functions
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        outputs = outputs.squeeze(1)  # binary
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    preds_list, targets_list = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze(1)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            preds_list.append(outputs.cpu())
            targets_list.append(y_batch.cpu())
    preds = torch.cat(preds_list)
    targets = torch.cat(targets_list)
    return total_loss / len(dataloader.dataset), preds, targets


In [79]:
# Train the model
epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_loader, optimizer, criterion, device)
    val_loss, preds, targets = eval_model(binary_model, test_loader, criterion, device)

    # Window-level predictions
    preds_label = (torch.sigmoid(preds) > 0.5).int()
    accuracy = (preds_label == targets.int()).float().mean()
    print(f"[Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Test Acc (window): {accuracy:.4f}")


[Epoch 1] Train Loss: 1.2882 | Test Acc (window): 0.9316
[Epoch 2] Train Loss: 1.0549 | Test Acc (window): 0.9159
[Epoch 3] Train Loss: 0.7758 | Test Acc (window): 0.9527
[Epoch 4] Train Loss: 0.5811 | Test Acc (window): 0.9741
[Epoch 5] Train Loss: 0.4762 | Test Acc (window): 0.9028
[Epoch 6] Train Loss: 0.4176 | Test Acc (window): 0.9632
[Epoch 7] Train Loss: 0.3636 | Test Acc (window): 0.9488
[Epoch 8] Train Loss: 0.3336 | Test Acc (window): 0.9041
[Epoch 9] Train Loss: 0.3312 | Test Acc (window): 0.9696
[Epoch 10] Train Loss: 0.2953 | Test Acc (window): 0.9725


In [80]:
# Patient-level evaluation
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# Aggregate window predictions per patient (max probability per patient)
test_patient_names = np.array(patient_names_clean)[test_mask]
df_pred = pd.DataFrame({
    "patient": test_patient_names,
    "pred_prob": torch.sigmoid(preds).numpy(),
    "true": targets.numpy()
})

# Max probability per patient
patient_pred = df_pred.groupby("patient")["pred_prob"].max()
patient_true = df_pred.groupby("patient")["true"].max()

# Patient-level threshold optimization (F1)
from sklearn.metrics import f1_score

best_f1, best_thresh = 0, 0.5
for t in np.linspace(0.1, 0.9, 9):
    f1 = f1_score(patient_true, (patient_pred > t).astype(int))
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

patient_preds_label = (patient_pred > best_thresh).astype(int)
print("Best patient-level threshold:", best_thresh, "F1:", best_f1)

print("Patient-level Confusion Matrix:")
print(confusion_matrix(patient_true, patient_preds_label))
print("Patient-level Classification Report:")
print(classification_report(patient_true, patient_preds_label))



Best patient-level threshold: 0.8 F1: 0.6153846153846154
Patient-level Confusion Matrix:
[[73  4]
 [ 1  4]]
Patient-level Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.95      0.97        77
         1.0       0.50      0.80      0.62         5

    accuracy                           0.94        82
   macro avg       0.74      0.87      0.79        82
weighted avg       0.96      0.94      0.95        82



### Good model

In [83]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# -------------------------------
# Assume you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# y_multilabel_clean: (N_windows, 5)
# window_ids_clean: list/array of window IDs
# patient_names_clean: list/array of patient_name per window
# -------------------------------

# Example:
# patient_names_clean = df_window['patient_name'].to_numpy()  

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train  = y_multilabel_clean[train_mask]
y_ml_test   = y_multilabel_clean[test_mask]
patient_names_test = patient_names_clean[test_mask]
window_ids_test = window_ids_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))
print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. Multi-label setup (abnormal windows only)
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

# -------------------------------
# 4. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_bin_loader = DataLoader(PoseDataset(X_train_stgcn, y_bin_train), batch_size=batch_size, shuffle=True)
test_bin_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=batch_size)
train_ml_loader  = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_ml_loader   = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

# -------------------------------
# 5. ST-GCN Model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool  = nn.AdaptiveAvgPool2d((1, num_joints))  # pool time dimension
        self.fc    = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        # x: (B, C, T, V)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

# -------------------------------
# 6. Training/Evaluation functions
# -------------------------------
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        if outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    all_logits, all_targets = [], []
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            if logits.shape[1] == 1:
                logits = logits.squeeze(1)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            all_logits.append(logits.cpu())
            all_targets.append(y_batch.cpu())
    preds = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    return total_loss / len(dataloader.dataset), preds, targets

# -------------------------------
# 7. Binary Model
# -------------------------------
num_joints = X_train_stgcn.shape[3]  # last dim = V

binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

# Weighted BCE
pos_weight = torch.tensor([(len(y_bin_train) - y_bin_train.sum()) / y_bin_train.sum()], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_bin_loader, optimizer, criterion, device)
    val_loss, preds, targets = eval_model(binary_model, test_bin_loader, criterion, device)
    preds_label = (1 / (1 + np.exp(-preds)) > 0.5).astype(int)
    accuracy = (preds_label == targets).mean()
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {accuracy:.4f}")

# -------------------------------
# 8. Patient-level evaluation
# -------------------------------
# Average window predictions per patient
test_patients_unique = np.unique(patient_names_test)
patient_preds = []
patient_true  = []

for p in test_patients_unique:
    mask = patient_names_test == p
    prob_avg = preds[mask].mean()
    patient_preds.append(prob_avg)
    patient_true.append(y_bin_test[mask][0])  # assume all windows have same label

patient_preds = np.array(patient_preds)
patient_true  = np.array(patient_true)

best_thresh = 0.5
patient_pred_labels = (patient_preds > best_thresh).astype(int)

print("Patient-level Confusion Matrix:")
print(confusion_matrix(patient_true, patient_pred_labels))
print("Patient-level Classification Report:")
print(classification_report(patient_true, patient_pred_labels, digits=4))


Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
Multi-label train windows: 460, test windows: 355
[Binary] Epoch 1/10 | Train Loss: 1.0709 | Test Acc: 0.9578
[Binary] Epoch 2/10 | Train Loss: 0.5252 | Test Acc: 0.9811
[Binary] Epoch 3/10 | Train Loss: 0.3221 | Test Acc: 0.9859
[Binary] Epoch 4/10 | Train Loss: 0.2568 | Test Acc: 0.9853
[Binary] Epoch 5/10 | Train Loss: 0.2172 | Test Acc: 0.9840
[Binary] Epoch 6/10 | Train Loss: 0.1902 | Test Acc: 0.9882
[Binary] Epoch 7/10 | Train Loss: 0.1915 | Test Acc: 0.9821
[Binary] Epoch 8/10 | Train Loss: 0.1692 | Test Acc: 0.9827
[Binary] Epoch 9/10 | Train Loss: 0.1610 | Test Acc: 0.9821
[Binary] Epoch 10/10 | Train Loss: 0.1337 | Test Acc: 0.9821
Patient-level Confusion Matrix:
[[77  0]
 [ 1  4]]
Patient-level Classification Report:
              precision    recall  f1-score   support

           0     0.9872    1.0000    0.9935        77
           1     1.0000    0.8000    0.88

## try to improve - too good on patient level - not trustworthy

In [84]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

# ---------------------------
# 1. Put the model in eval mode
# ---------------------------
binary_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

all_probs = []
all_targets = []
all_patient_names = []

with torch.no_grad():
    for X_batch, y_batch in test_bin_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = binary_model(X_batch).squeeze(1)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())
        all_patient_names.extend(patient_names_test[:len(probs)])  # align patient names

# Concatenate arrays
y_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)
patient_names_array = np.array(all_patient_names)

# ---------------------------
# 2. Window-level metrics
# ---------------------------
y_pred_window = (y_probs > 0.5).astype(int)  # default threshold

print("Window-level Metrics:")
print(f"Accuracy:  {accuracy_score(y_true, y_pred_window):.4f}")
print(f"Precision: {precision_score(y_true, y_pred_window, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred_window, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred_window, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_true, y_probs):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_window))

# ---------------------------
# 3. Patient-level aggregation
# ---------------------------
# Aggregate by patient: patient is abnormal if any window is predicted abnormal
patients = np.unique(patient_names_array)
y_true_patient = []
y_pred_patient = []

for p in patients:
    mask = patient_names_array == p
    y_true_patient.append(y_true[mask].max())      # 1 if any window abnormal
    y_pred_patient.append(y_pred_window[mask].max())  # same for prediction

y_true_patient = np.array(y_true_patient)
y_pred_patient = np.array(y_pred_patient)

print("\nPatient-level Metrics:")
print(f"Accuracy:  {accuracy_score(y_true_patient, y_pred_patient):.4f}")
print(f"Precision: {precision_score(y_true_patient, y_pred_patient, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true_patient, y_pred_patient, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_true_patient, y_pred_patient, zero_division=0):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_true_patient, y_pred_patient))
print("\nClassification Report:")
print(classification_report(y_true_patient, y_pred_patient, zero_division=0))

# ---------------------------
# 4. Optional: Find best patient-level threshold
# ---------------------------
thresholds = np.arange(0.1, 1.0, 0.05)
best_f1 = 0
best_thresh = 0.5

for t in thresholds:
    y_pred_patient_thresh = np.array([
        (y_probs[patient_names_array == p].max() > t).astype(int)
        for p in patients
    ])
    f1 = f1_score(y_true_patient, y_pred_patient_thresh, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\nBest patient-level threshold: {best_thresh:.2f}, F1: {best_f1:.4f}")


Window-level Metrics:
Accuracy:  0.9821
Precision: 0.8824
Recall:    0.9718
F1 Score:  0.9249
ROC-AUC:   0.9951
Confusion Matrix:
[[2726   46]
 [  10  345]]

Patient-level Metrics:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000
Confusion Matrix:
[[7]]

Classification Report:
              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00         7

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7


Best patient-level threshold: 0.10, F1: 1.0000


## Last final version

In [85]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# -------------------------------
# Assume after QC you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# window_ids_clean: list of window IDs
# patient_names_clean: list/array of patient_name per window
# -------------------------------

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
patient_names_test = patient_names_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_loader = DataLoader(PoseDataset(X_train_stgcn, y_bin_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=batch_size)

# -------------------------------
# 4. ST-GCN Model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool  = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc    = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        # x: (B, C, T, J)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)  # (B, 128, 1, J)
        x = x.flatten(1)  # (B, 128*J)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_train_stgcn.shape[3]
binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

# -------------------------------
# 5. Class-weighted BCE
# -------------------------------
pos_weight = torch.tensor([(len(y_bin_train) - y_bin_train.sum()) / y_bin_train.sum()], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# 6. Training loop
# -------------------------------
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze(1)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    all_logits, all_targets = [], []
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze(1)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            all_logits.append(outputs.cpu())
            all_targets.append(y_batch.cpu())
    logits = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return total_loss / len(dataloader.dataset), preds, targets, probs

# -------------------------------
# 7. Train for multiple epochs
# -------------------------------
epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_loader, optimizer, criterion, device)
    val_loss, preds, targets, probs = eval_model(binary_model, test_loader, criterion, device)
    acc = accuracy_score(targets, preds)
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {acc:.4f}")

# -------------------------------
# 8. Window-level and Patient-level evaluation
# -------------------------------
# Window-level metrics
acc_win  = accuracy_score(targets, preds)
prec_win = precision_score(targets, preds)
rec_win  = recall_score(targets, preds)
f1_win  = f1_score(targets, preds)
roc_win = roc_auc_score(targets, probs)
cm_win = confusion_matrix(targets, preds)
print("\nWindow-level Metrics:")
print(f"Accuracy:  {acc_win:.4f}")
print(f"Precision: {prec_win:.4f}")
print(f"Recall:    {rec_win:.4f}")
print(f"F1 Score:  {f1_win:.4f}")
print(f"ROC-AUC:   {roc_win:.4f}")
print("Confusion Matrix:")
print(cm_win)

# Patient-level metrics
# Aggregate windows per patient
patient_pred_probs = {}
for p_name, prob in zip(patient_names_test, probs):
    patient_pred_probs.setdefault(p_name, []).append(prob)

# Mean probability per patient
patient_probs = np.array([np.mean(v) for v in patient_pred_probs.values()])
patient_names_unique = np.array(list(patient_pred_probs.keys()))

# Choose threshold (default 0.5 or can optimize for F1)
threshold = 0.5
patient_preds = (patient_probs > threshold).astype(int)

# True labels per patient (assume any abnormal window => abnormal patient)
patient_targets = []
for p_name in patient_names_unique:
    mask = patient_names_test == p_name
    patient_targets.append(int(np.any(y_bin_test[mask])))
patient_targets = np.array(patient_targets)

acc_pat  = accuracy_score(patient_targets, patient_preds)
prec_pat = precision_score(patient_targets, patient_preds, zero_division=0)
rec_pat  = recall_score(patient_targets, patient_preds, zero_division=0)
f1_pat  = f1_score(patient_targets, patient_preds, zero_division=0)
cm_pat = confusion_matrix(patient_targets, patient_preds)
report_pat = classification_report(patient_targets, patient_preds, zero_division=0)

print("\nPatient-level Metrics:")
print(f"Accuracy:  {acc_pat:.4f}")
print(f"Precision: {prec_pat:.4f}")
print(f"Recall:    {rec_pat:.4f}")
print(f"F1 Score:  {f1_pat:.4f}")
print("Confusion Matrix:")
print(cm_pat)
print("\nClassification Report:")
print(report_pat)


Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
[Binary] Epoch 1/10 | Train Loss: 1.0812 | Test Acc: 0.9667
[Binary] Epoch 2/10 | Train Loss: 0.4977 | Test Acc: 0.9571
[Binary] Epoch 3/10 | Train Loss: 0.3225 | Test Acc: 0.9856
[Binary] Epoch 4/10 | Train Loss: 0.2452 | Test Acc: 0.9527
[Binary] Epoch 5/10 | Train Loss: 0.1953 | Test Acc: 0.9843
[Binary] Epoch 6/10 | Train Loss: 0.1784 | Test Acc: 0.9859
[Binary] Epoch 7/10 | Train Loss: 0.1576 | Test Acc: 0.9885
[Binary] Epoch 8/10 | Train Loss: 0.1517 | Test Acc: 0.9776
[Binary] Epoch 9/10 | Train Loss: 0.1352 | Test Acc: 0.9741
[Binary] Epoch 10/10 | Train Loss: 0.1209 | Test Acc: 0.9805

Window-level Metrics:
Accuracy:  0.9805
Precision: 0.8910
Recall:    0.9437
F1 Score:  0.9166
ROC-AUC:   0.9944
Confusion Matrix:
[[2731   41]
 [  20  335]]

Patient-level Metrics:
Accuracy:  0.9878
Precision: 1.0000
Recall:    0.8000
F1 Score:  0.8889
Confusion Matrix:
[[77  0]
 [ 1  4

In [ ]:
#patient level threshold optimization
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# ---------------------------
# 1. Predict window-level probabilities
# ---------------------------
binary_model.eval()
all_probs = []
all_targets = []
patient_names_test = patient_names_clean[test_mask]  # ensure this matches test windows

with torch.no_grad():
    for X_batch, y_batch in test_bin_loader:
        X_batch = X_batch.to(device)
        logits = binary_model(X_batch).squeeze(1)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu())
        all_targets.append(y_batch.cpu())

window_probs = torch.cat(all_probs).numpy()        # shape: (N_test_windows,)
window_targets = torch.cat(all_targets).numpy()    # shape: (N_test_windows,)

# ---------------------------
# 2. Optimize patient-level threshold
# ---------------------------
unique_patients = np.unique(patient_names_test)
best_f1 = 0
best_threshold = 0.5  # default
thresholds = np.arange(0.05, 1.0, 0.05)

for th in thresholds:
    patient_preds = []
    patient_targets = []
    for p in unique_patients:
        # all windows for this patient
        idx = patient_names_test == p
        p_prob = window_probs[idx].mean()  # aggregate: mean probability over windows
        p_pred = int(p_prob >= th)
        p_target = int(window_targets[idx].max())  # 1 if any window abnormal
        patient_preds.append(p_pred)
        patient_targets.append(p_target)
    
    f1 = f1_score(patient_targets, patient_preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = th

print(f"Best patient-level threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

# ---------------------------
# 3. Compute patient-level metrics at best threshold
# ---------------------------
patient_preds = []
patient_targets = []

for p in unique_patients:
    idx = patient_names_test == p
    p_prob = window_probs[idx].mean()
    p_pred = int(p_prob >= best_threshold)
    p_target = int(window_targets[idx].max())
    patient_preds.append(p_pred)
    patient_targets.append(p_target)

cm = confusion_matrix(patient_targets, patient_preds)
report = classification_report(patient_targets, patient_preds, digits=4)
accuracy = np.mean(np.array(patient_preds) == np.array(patient_targets))

print("Patient-level Confusion Matrix:")
print(cm)
print("Patient-level Classification Report:")
print(report)
print(f"Patient-level Accuracy: {accuracy:.4f}")


Best patient-level threshold: 0.35, F1: 0.9091
Patient-level Confusion Matrix:
[[76  1]
 [ 0  5]]
Patient-level Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9870    0.9935        77
           1     0.8333    1.0000    0.9091         5

    accuracy                         0.9878        82
   macro avg     0.9167    0.9935    0.9513        82
weighted avg     0.9898    0.9878    0.9883        82

Patient-level Accuracy: 0.9878


# Final in one binary

In [90]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

# -------------------------------
# 0. Assumes these exist from your preprocessing
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# window_ids_clean: list of window IDs
# patient_names_clean: array/list of patient names per window
# -------------------------------

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
patient_names_test = patient_names_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_loader = DataLoader(PoseDataset(X_train_stgcn, y_bin_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=batch_size)

# -------------------------------
# 4. Simple ST-GCN Model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool  = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc    = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        # x: (B, C, T, J)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)       # (B, 128, 1, J)
        x = x.flatten(1)       # (B, 128*J)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_train_stgcn.shape[3]
binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

# -------------------------------
# 5. Class-weighted BCE Loss
# -------------------------------
pos_weight = torch.tensor([(len(y_bin_train) - y_bin_train.sum()) / y_bin_train.sum()], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# 6. Training / Evaluation Functions
# -------------------------------
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze(1)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    all_logits, all_targets = [], []
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze(1)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            all_logits.append(outputs.cpu())
            all_targets.append(y_batch.cpu())
    logits = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return total_loss / len(dataloader.dataset), preds, targets, probs

# -------------------------------
# 7. Train Loop
# -------------------------------
epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_loader, optimizer, criterion, device)
    val_loss, preds, targets, probs = eval_model(binary_model, test_loader, criterion, device)
    acc = accuracy_score(targets, preds)
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {acc:.4f}")

# -------------------------------
# 8. Window-level metrics
# -------------------------------
window_accuracy  = accuracy_score(targets, preds)
window_precision = precision_score(targets, preds, zero_division=0)
window_recall    = recall_score(targets, preds, zero_division=0)
window_f1        = f1_score(targets, preds, zero_division=0)
window_roc_auc   = roc_auc_score(targets, probs)
window_cm        = confusion_matrix(targets, preds)

print("\nWindow-level Metrics:")
print(f"Accuracy:  {window_accuracy:.4f}")
print(f"Precision: {window_precision:.4f}")
print(f"Recall:    {window_recall:.4f}")
print(f"F1 Score:  {window_f1:.4f}")
print(f"ROC-AUC:   {window_roc_auc:.4f}")
print("Confusion Matrix:\n", window_cm)

# -------------------------------
# 9. Patient-level metrics (threshold optimization)
# -------------------------------
unique_patients = np.unique(patient_names_test)
best_f1 = 0
best_threshold = 0.5
thresholds = np.arange(0.05, 1.0, 0.05)

for th in thresholds:
    patient_preds, patient_targets = [], []
    for p in unique_patients:
        idx = patient_names_test == p
        p_prob = probs[idx].mean()
        p_pred = int(p_prob >= th)
        p_target = int(targets[idx].max())
        patient_preds.append(p_pred)
        patient_targets.append(p_target)
    f1 = f1_score(patient_targets, patient_preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = th

print(f"\nBest patient-level threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

# Compute metrics at best threshold
patient_preds, patient_targets = [], []
for p in unique_patients:
    idx = patient_names_test == p
    p_prob = probs[idx].mean()
    p_pred = int(p_prob >= best_threshold)
    p_target = int(targets[idx].max())
    patient_preds.append(p_pred)
    patient_targets.append(p_target)

patient_accuracy = np.mean(np.array(patient_preds) == np.array(patient_targets))
patient_cm = confusion_matrix(patient_targets, patient_preds)
patient_report = classification_report(patient_targets, patient_preds, digits=4)

print("\nPatient-level Metrics:")
print(f"Accuracy: {patient_accuracy:.4f}")
print("Confusion Matrix:\n", patient_cm)
print("Classification Report:\n", patient_report)


Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
[Binary] Epoch 1/10 | Train Loss: 1.0929 | Test Acc: 0.9635
[Binary] Epoch 2/10 | Train Loss: 0.5475 | Test Acc: 0.9543
[Binary] Epoch 3/10 | Train Loss: 0.3175 | Test Acc: 0.9434
[Binary] Epoch 4/10 | Train Loss: 0.2346 | Test Acc: 0.9818
[Binary] Epoch 5/10 | Train Loss: 0.1962 | Test Acc: 0.9885
[Binary] Epoch 6/10 | Train Loss: 0.1660 | Test Acc: 0.9770
[Binary] Epoch 7/10 | Train Loss: 0.1390 | Test Acc: 0.9677
[Binary] Epoch 8/10 | Train Loss: 0.1372 | Test Acc: 0.9904
[Binary] Epoch 9/10 | Train Loss: 0.1216 | Test Acc: 0.9789
[Binary] Epoch 10/10 | Train Loss: 0.1224 | Test Acc: 0.9280

Window-level Metrics:
Accuracy:  0.9280
Precision: 0.6128
Recall:    0.9944
F1 Score:  0.7583
ROC-AUC:   0.9953
Confusion Matrix:
 [[2549  223]
 [   2  353]]

Best patient-level threshold: 0.80, F1: 0.8889

Patient-level Metrics:
Accuracy: 0.9878
Confusion Matrix:
 [[77  0]
 [ 1  4]]
Cl

In [91]:
# export the model as a bin file
# Assuming your model is called binary_model
import torch

# Option 1: Save the state_dict (recommended)
torch.save(binary_model.state_dict(), "../models/binary_model.bin")
print("Model weights saved to binary_model.bin")

# Option 2: Save the full model (include structure, less portable)
torch.save(binary_model, "../models/binary_model_full.bin")
print("Full model saved to binary_model_full.bin")


Model weights saved to binary_model.bin
Full model saved to binary_model_full.bin


# Multi-label prediction

In [94]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

# -------------------------------
# Assume after QC you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,) -> 0=normal, 1=abnormal
# y_multilabel_clean: (N_windows, 5) -> abnormal type labels
# window_ids_clean: list of window IDs
# patient_names_clean: list/array of patient_name per window
# -------------------------------

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train = y_multilabel_clean[train_mask]
y_ml_test  = y_multilabel_clean[test_mask]
patient_names_test = patient_names_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))
print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. Filter only abnormal windows for multi-label model
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

patient_names_ml_test = patient_names_test[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

# -------------------------------
# 4. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_loader = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

# -------------------------------
# 5. ST-GCN Model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_ml_train_stgcn.shape[3]
multi_model = SimpleSTGCN(num_joints=num_joints, out_classes=5).to(device)
optimizer = torch.optim.Adam(multi_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

# -------------------------------
# 6. Training loop
# -------------------------------
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    all_logits, all_targets = [], []
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            all_logits.append(outputs.cpu())
            all_targets.append(y_batch.cpu())
    logits = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return total_loss / len(dataloader.dataset), preds, targets, probs

# -------------------------------
# 7. Train the multi-label model
# -------------------------------
epochs = 10
for epoch in range(epochs):
    train_loss = train_model(multi_model, train_loader, optimizer, criterion, device)
    val_loss, preds, targets, probs = eval_model(multi_model, test_loader, criterion, device)
    print(f"[Multi-label] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {val_loss:.4f}")

# -------------------------------
# 8. Window-level evaluation
# -------------------------------
for i in range(probs.shape[1]):
    print(f"\nLabel {i+1} window-level metrics:")
    print(f"Accuracy: {accuracy_score(targets[:, i], preds[:, i]):.4f}")
    print(f"Precision: {precision_score(targets[:, i], preds[:, i], zero_division=0):.4f}")
    print(f"Recall: {recall_score(targets[:, i], preds[:, i], zero_division=0):.4f}")
    print(f"F1: {f1_score(targets[:, i], preds[:, i], zero_division=0):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(targets[:, i], preds[:, i])}")

# -------------------------------
# 9. Patient-level evaluation (only abnormal patients)
# -------------------------------
unique_patients = np.unique(patient_names_ml_test)
patient_preds = []
patient_targets = []

for p in unique_patients:
    idx = patient_names_ml_test == p
    p_prob = probs[idx].mean(axis=0)
    p_pred = (p_prob > 0.5).astype(int)
    p_target = targets[idx].max(axis=0)
    patient_preds.append(p_pred)
    patient_targets.append(p_target)

patient_preds = np.array(patient_preds)
patient_targets = np.array(patient_targets)

for i in range(patient_preds.shape[1]):
    print(f"\nLabel {i+1} patient-level metrics:")
    print(f"Accuracy: {accuracy_score(patient_targets[:, i], patient_preds[:, i]):.4f}")
    print(f"Precision: {precision_score(patient_targets[:, i], patient_preds[:, i], zero_division=0):.4f}")
    print(f"Recall: {recall_score(patient_targets[:, i], patient_preds[:, i], zero_division=0):.4f}")
    print(f"F1: {f1_score(patient_targets[:, i], patient_preds[:, i], zero_division=0):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(patient_targets[:, i], patient_preds[:, i])}")

# -------------------------------
# 10. Save model
# -------------------------------
torch.save(multi_model.state_dict(), "multi_label_model.bin")
print("Multi-label model saved as 'multi_label_model.bin'")


Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
Multi-label train windows: 460, test windows: 355
[Multi-label] Epoch 1/10 | Train Loss: 0.5268 | Test Loss: 0.7411
[Multi-label] Epoch 2/10 | Train Loss: 0.4668 | Test Loss: 0.6587
[Multi-label] Epoch 3/10 | Train Loss: 0.4534 | Test Loss: 0.6735
[Multi-label] Epoch 4/10 | Train Loss: 0.4434 | Test Loss: 0.6906
[Multi-label] Epoch 5/10 | Train Loss: 0.4316 | Test Loss: 0.7258
[Multi-label] Epoch 6/10 | Train Loss: 0.4168 | Test Loss: 0.7334
[Multi-label] Epoch 7/10 | Train Loss: 0.4016 | Test Loss: 0.7516
[Multi-label] Epoch 8/10 | Train Loss: 0.3895 | Test Loss: 0.8080
[Multi-label] Epoch 9/10 | Train Loss: 0.3814 | Test Loss: 0.7861
[Multi-label] Epoch 10/10 | Train Loss: 0.3731 | Test Loss: 0.8684

Label 1 window-level metrics:
Accuracy: 0.8000
Precision: 0.0714
Recall: 0.0426
F1: 0.0533
Confusion Matrix:
[[282  26]
 [ 45   2]]

Label 2 window-level metrics:
Accuracy: 0.63

In [95]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# -------------------------------
# Assume after QC you have:
# X_clean, y_binary_clean, y_multilabel_clean, window_ids_clean, patient_names_clean
# -------------------------------

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = np.random.RandomState(42).choice(unique_patients, size=len(unique_patients), replace=False).tolist(), unique_patients.tolist()

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train = y_multilabel_clean[train_mask]
y_ml_test  = y_multilabel_clean[test_mask]
patient_names_test = patient_names_clean[test_mask]

# -------------------------------
# 2. Transpose to ST-GCN format
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

# -------------------------------
# 3. Keep only abnormal windows for multi-label training
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

# -------------------------------
# 4. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_ml_loader = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_ml_loader  = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

# -------------------------------
# 5. ST-GCN model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool  = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc    = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_ml_train_stgcn.shape[3]
multi_model = SimpleSTGCN(num_joints=num_joints, out_classes=5).to(device)

# -------------------------------
# 6. Compute per-label pos_weight for BCEWithLogitsLoss
# -------------------------------
pos_weight = []
for i in range(y_ml_train_filtered.shape[1]):
    n_pos = y_ml_train_filtered[:, i].sum()
    n_neg = len(y_ml_train_filtered) - n_pos
    pw = (n_neg / n_pos) if n_pos > 0 else 1.0
    pos_weight.append(pw)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(multi_model.parameters(), lr=1e-3)

# -------------------------------
# 7. Training loop
# -------------------------------
epochs = 10

for epoch in range(epochs):
    multi_model.train()
    total_loss = 0
    for X_batch, y_batch in train_ml_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = multi_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    train_loss = total_loss / len(train_ml_loader.dataset)

    # Eval
    multi_model.eval()
    val_loss = 0
    all_probs, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_ml_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = multi_model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            all_probs.append(torch.sigmoid(outputs).cpu())
            all_targets.append(y_batch.cpu())
    val_loss /= len(test_ml_loader.dataset)
    probs = torch.cat(all_probs).numpy()
    targets = torch.cat(all_targets).numpy()
    print(f"[Multi-label] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {val_loss:.4f}")

# -------------------------------
# 8. Window-level metrics per label
# -------------------------------
for i in range(targets.shape[1]):
    preds = (probs[:, i] > 0.5).astype(int)
    acc = (preds == targets[:, i]).mean()
    f1 = f1_score(targets[:, i], preds, zero_division=0)
    cm = confusion_matrix(targets[:, i], preds)
    print(f"Label {i+1} window-level metrics:")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print()

# -------------------------------
# 9. Patient-level metrics per label
# -------------------------------
patient_names_test_ab = patient_names_test[y_bin_test==1]  # abnormal only
unique_patients = np.unique(patient_names_test_ab)

for i in range(targets.shape[1]):
    patient_preds, patient_targets = [], []
    for p in unique_patients:
        idx = patient_names_test_ab == p
        p_prob = probs[idx, i].mean()
        p_pred = int(p_prob >= 0.5)
        p_target = int(targets[idx, i].max())
        patient_preds.append(p_pred)
        patient_targets.append(p_target)
    cm = confusion_matrix(patient_targets, patient_preds)
    f1 = f1_score(patient_targets, patient_preds, zero_division=0)
    print(f"Label {i+1} patient-level metrics:")
    print(f"F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print()

# -------------------------------
# 10. Save the model
# -------------------------------
torch.save(multi_model.state_dict(), "multi_label_abnormal_model.bin")
print("Multi-label model saved as 'multi_label_abnormal_model.bin'")


Multi-label train windows: 815, test windows: 815
[Multi-label] Epoch 1/10 | Train Loss: 1.0874 | Test Loss: 1.0688
[Multi-label] Epoch 2/10 | Train Loss: 1.0530 | Test Loss: 1.0175
[Multi-label] Epoch 3/10 | Train Loss: 1.0062 | Test Loss: 0.9691
[Multi-label] Epoch 4/10 | Train Loss: 0.9546 | Test Loss: 0.9319
[Multi-label] Epoch 5/10 | Train Loss: 0.9278 | Test Loss: 0.9140
[Multi-label] Epoch 6/10 | Train Loss: 0.9107 | Test Loss: 0.8894
[Multi-label] Epoch 7/10 | Train Loss: 0.8890 | Test Loss: 0.8809
[Multi-label] Epoch 8/10 | Train Loss: 0.8786 | Test Loss: 0.8665
[Multi-label] Epoch 9/10 | Train Loss: 0.8707 | Test Loss: 0.8582
[Multi-label] Epoch 10/10 | Train Loss: 0.8549 | Test Loss: 0.8493
Label 1 window-level metrics:
Accuracy: 0.7583 | F1: 0.5513
Confusion Matrix:
[[497 161]
 [ 36 121]]

Label 2 window-level metrics:
Accuracy: 0.7448 | F1: 0.5517
Confusion Matrix:
[[479 155]
 [ 53 128]]

Label 3 window-level metrics:
Accuracy: 0.7411 | F1: 0.5443
Confusion Matrix:
[[478 1

In [97]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# -------------------------------
# Assume after QC you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# y_multilabel_clean: (N_windows, num_labels)
# window_ids_clean: list of window IDs
# patient_names_clean: list/array of patient_name per window
# -------------------------------

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train = y_multilabel_clean[train_mask]
y_ml_test  = y_multilabel_clean[test_mask]
patient_names_test_full = patient_names_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. Separate abnormal windows for multi-label model
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

# Filter patient names to match abnormal windows
patient_names_test = patient_names_test_full[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

# -------------------------------
# 4. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_ml_loader = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_ml_loader  = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

# -------------------------------
# 5. ST-GCN Model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool  = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc    = nn.Linear(128 * num_joints, out_classes)

    def forward(self, x):
        # x: (B, C, T, J)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)  # (B, 128, 1, J)
        x = x.flatten(1)  # (B, 128*J)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_ml_train_stgcn.shape[3]
num_labels = y_ml_train_filtered.shape[1]

multi_model = SimpleSTGCN(num_joints=num_joints, out_classes=num_labels).to(device)
optimizer = torch.optim.Adam(multi_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

# -------------------------------
# 6. Training loop
# -------------------------------
epochs = 10

for epoch in range(epochs):
    # Train
    multi_model.train()
    total_loss = 0
    for X_batch, y_batch in train_ml_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = multi_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    train_loss = total_loss / len(train_ml_loader.dataset)

    # Eval
    multi_model.eval()
    total_loss = 0
    all_logits, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_ml_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = multi_model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            all_logits.append(outputs.cpu())
            all_targets.append(y_batch.cpu())
    val_loss = total_loss / len(test_ml_loader.dataset)
    all_logits = torch.cat(all_logits).numpy()
    all_targets = torch.cat(all_targets).numpy()
    print(f"[Multi-label] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {val_loss:.4f}")

# -------------------------------
# 7. Window-level metrics
# -------------------------------
probs = 1 / (1 + np.exp(-all_logits))
preds = (probs > 0.5).astype(int)

for label_idx in range(num_labels):
    acc = accuracy_score(all_targets[:, label_idx], preds[:, label_idx])
    f1 = f1_score(all_targets[:, label_idx], preds[:, label_idx], zero_division=0)
    cm = confusion_matrix(all_targets[:, label_idx], preds[:, label_idx])
    print(f"Label {label_idx+1} window-level metrics:")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}\n")

# -------------------------------
# 8. Patient-level metrics
# -------------------------------
best_thresholds = {}
unique_patients = np.unique(patient_names_test)

for label_idx in range(num_labels):
    best_f1 = 0
    best_th = 0.5
    thresholds = np.arange(0.05, 1.0, 0.05)
    for th in thresholds:
        patient_preds, patient_targets = [], []
        for p in unique_patients:
            idx = patient_names_test == p
            p_prob = probs[idx, label_idx].max()
            p_pred = int(p_prob >= th)
            p_target = int(all_targets[idx, label_idx].max())
            patient_preds.append(p_pred)
            patient_targets.append(p_target)
        f1 = f1_score(patient_targets, patient_preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
    best_thresholds[label_idx] = best_th
    print(f"Label {label_idx+1} best patient-level threshold: {best_th:.2f}, F1: {best_f1:.4f}")

# -------------------------------
# 9. Save model
# -------------------------------
torch.save(multi_model.state_dict(), "multi_label_abnormal_model.bin")
print("Multi-label model saved as 'multi_label_abnormal_model.bin'")


Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
Multi-label train windows: 460, test windows: 355
[Multi-label] Epoch 1/10 | Train Loss: 0.5165 | Test Loss: 0.7414
[Multi-label] Epoch 2/10 | Train Loss: 0.4691 | Test Loss: 0.6722
[Multi-label] Epoch 3/10 | Train Loss: 0.4535 | Test Loss: 0.6489
[Multi-label] Epoch 4/10 | Train Loss: 0.4387 | Test Loss: 0.7013
[Multi-label] Epoch 5/10 | Train Loss: 0.4211 | Test Loss: 0.7025
[Multi-label] Epoch 6/10 | Train Loss: 0.4007 | Test Loss: 0.7102
[Multi-label] Epoch 7/10 | Train Loss: 0.3857 | Test Loss: 0.8170
[Multi-label] Epoch 8/10 | Train Loss: 0.3737 | Test Loss: 0.7880
[Multi-label] Epoch 9/10 | Train Loss: 0.3637 | Test Loss: 0.8194
[Multi-label] Epoch 10/10 | Train Loss: 0.3552 | Test Loss: 0.8621
Label 1 window-level metrics:
Accuracy: 0.8254 | F1: 0.0000
Confusion Matrix:
[[293  15]
 [ 47   0]]

Label 2 window-level metrics:
Accuracy: 0.6225 | F1: 0.0694
Confusion Matrix

# Best models for binary and multi-label (on abnormal dataset only)

In [98]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# -------------------------------
# Assume after QC you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# y_multilabel_clean: (N_windows, num_labels)
# window_ids_clean
# patient_names_clean
# -------------------------------

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train = y_multilabel_clean[train_mask]
y_ml_test = y_multilabel_clean[test_mask]
patient_names_test = patient_names_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))
print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. Separate abnormal windows for multi-label
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

# -------------------------------
# 4. PyTorch Dataset
# -------------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
# Binary
train_bin_loader = DataLoader(PoseDataset(X_train_stgcn, y_bin_train), batch_size=batch_size, shuffle=True)
test_bin_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=batch_size)
# Multi-label
train_ml_loader = DataLoader(PoseDataset(X_ml_train_stgcn, y_ml_train_filtered), batch_size=batch_size, shuffle=True)
test_ml_loader  = DataLoader(PoseDataset(X_ml_test_stgcn, y_ml_test_filtered), batch_size=batch_size)

# -------------------------------
# 5. ST-GCN Model
# -------------------------------
class SimpleSTGCN(nn.Module):
    def __init__(self, num_joints, in_channels=3, out_classes=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1,1))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(1,1))
        self.pool  = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc    = nn.Linear(128 * num_joints, out_classes)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_joints = X_train_stgcn.shape[3]

# Binary Model
binary_model = SimpleSTGCN(num_joints=num_joints, out_classes=1).to(device)
optimizer_bin = torch.optim.Adam(binary_model.parameters(), lr=1e-3)
pos_weight = torch.tensor([(len(y_bin_train)-y_bin_train.sum())/y_bin_train.sum()], device=device)
criterion_bin = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Multi-label Model
num_labels = y_ml_train_filtered.shape[1]
multi_model = SimpleSTGCN(num_joints=num_joints, out_classes=num_labels).to(device)
optimizer_ml = torch.optim.Adam(multi_model.parameters(), lr=1e-3)
criterion_ml = nn.BCEWithLogitsLoss()

# -------------------------------
# 6. Training function
# -------------------------------
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        if outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def eval_model(model, dataloader, criterion, device):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            if outputs.shape[1] == 1:
                outputs = outputs.squeeze(1)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()
            all_preds.append(preds.cpu())
            all_targets.append(y_batch.cpu())
            all_probs.append(probs.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    all_probs = torch.cat(all_probs).numpy()
    return total_loss / len(dataloader.dataset), all_preds, all_targets, all_probs

# -------------------------------
# 7. Train binary model
# -------------------------------
epochs = 10
for epoch in range(epochs):
    train_loss = train_model(binary_model, train_bin_loader, optimizer_bin, criterion_bin, device)
    val_loss, preds, targets, probs = eval_model(binary_model, test_bin_loader, criterion_bin, device)
    acc = accuracy_score(targets, preds)
    print(f"[Binary] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {acc:.4f}")

# -------------------------------
# 8. Patient-level threshold optimization (binary)
# -------------------------------
unique_patients_test = np.unique(patient_names_test)
best_f1 = 0
best_threshold = 0.5
for th in np.arange(0.05, 1.0, 0.05):
    p_preds, p_targets = [], []
    for p in unique_patients_test:
        idx = patient_names_test == p
        p_prob = probs[idx].mean()
        p_pred = int(p_prob >= th)
        p_target = int(targets[idx].max())
        p_preds.append(p_pred)
        p_targets.append(p_target)
    f1 = f1_score(p_targets, p_preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = th
print(f"Best patient-level threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

# Patient-level metrics
p_preds, p_targets = [], []
for p in unique_patients_test:
    idx = patient_names_test == p
    p_prob = probs[idx].mean()
    p_pred = int(p_prob >= best_threshold)
    p_target = int(targets[idx].max())
    p_preds.append(p_pred)
    p_targets.append(p_target)

cm = confusion_matrix(p_targets, p_preds)
report = classification_report(p_targets, p_preds, digits=4)
accuracy = np.mean(np.array(p_preds) == np.array(p_targets))
print("Patient-level Confusion Matrix:")
print(cm)
print("Patient-level Classification Report:")
print(report)
print(f"Patient-level Accuracy: {accuracy:.4f}")

# Save binary model
torch.save(binary_model.state_dict(), "binary_model.bin")

# -------------------------------
# 9. Train multi-label model on abnormal windows
# -------------------------------
for epoch in range(epochs):
    train_loss = train_model(multi_model, train_ml_loader, optimizer_ml, criterion_ml, device)
    val_loss, preds, targets, probs = eval_model(multi_model, test_ml_loader, criterion_ml, device)
    print(f"[Multi-label] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {val_loss:.4f}")

# Save multi-label model
torch.save(multi_model.state_dict(), "multi_label_abnormal_model.bin")


Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14), test: (3127, 3, 60, 14)
Multi-label train windows: 460, test windows: 355
[Binary] Epoch 1/10 | Train Loss: 1.1018 | Test Acc: 0.9492
[Binary] Epoch 2/10 | Train Loss: 0.6062 | Test Acc: 0.9651
[Binary] Epoch 3/10 | Train Loss: 0.3418 | Test Acc: 0.9840
[Binary] Epoch 4/10 | Train Loss: 0.2402 | Test Acc: 0.9843
[Binary] Epoch 5/10 | Train Loss: 0.2444 | Test Acc: 0.9552
[Binary] Epoch 6/10 | Train Loss: 0.1735 | Test Acc: 0.9862
[Binary] Epoch 7/10 | Train Loss: 0.1650 | Test Acc: 0.9885
[Binary] Epoch 8/10 | Train Loss: 0.1510 | Test Acc: 0.9741
[Binary] Epoch 9/10 | Train Loss: 0.1349 | Test Acc: 0.9808
[Binary] Epoch 10/10 | Train Loss: 0.1291 | Test Acc: 0.9709
Best patient-level threshold: 0.55, F1: 0.8889
Patient-level Confusion Matrix:
[[77  0]
 [ 1  4]]
Patient-level Classification Report:
              precision    recall  f1-score   support

           0     0.9872    1.0000    0.9935     